# **News Summarization Application using LangChain (Labwork 3)**

---

This project implements a **News Summarization Application** using the LangChain framework.  
The system retrieves news articles based on user queries, processes them using semantic embeddings, and generates customized summaries according to user preferences.

The application integrates multiple AI and NLP technologies including:

- **NewsAPI** for retrieving real-time news articles through the "everything" endpoint.
- **Sentence Transformers** specifically the **all-MiniLM-L6-v2** model for generating semantic embeddings
- **Chroma Vector Database** for storing and retrieving document embeddings
- **Groq LLM (Kimi K2 model)** for generating summaries
- **LangChain Summarization Chains** for orchestrating LLM workflows

The system allows users to:
- Search for news articles on specific topics
- Generate **brief or detailed summaries**
- Store topics of interest
- Track search history



This project demonstrates how **retrieval augmented generation (RAG)** can be applied to build intelligent information systems.

In [ ]:
!pip install -qU langchain langchain-community langchain-groq langchain-huggingface langchain-chroma sentence-transformers requests opentelemetry-api opentelemetry-sdk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51

## **API Key Configuration**

This application requires two external APIs:

### 1. **NewsAPI**
Used to retrieve real-time news articles using the **Everything endpoint**.

Users can register for a free key at:
https://newsapi.org/register


### 2. **Groq API**
Used to access the **Kimi K2 language model** for generating summaries.

To protect sensitive credentials, the keys are loaded securely using environment variables instead of hardcoding them directly into the source code.

In [ ]:
import os
import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
if "NEWS_API_KEY" not in os.environ:
    os.environ["NEWS_API_KEY"] = getpass.getpass("Enter your News API key: ")

Enter your Groq API key: ··········
Enter your News API key: ··········


In [ ]:
# news_api_key: set via NEWS_API_KEY env variable
# groq_api_key: set via GROQ_API_KEY env variable

In [ ]:
{
  "custom_models": [
    {
      "model_display_name": "Kimi K2 [Groq]",
      "model": "moonshotai/kimi-k2-instruct-0905",
      "base_url": "https://api.groq.com/openai/v1",
      "api_key": "your_groq_api_key_here",
      "provider": "generic-chat-completion-api",
      "max_tokens": 16384
    }
  ]
}

## **News Retrieval Module**

The **NewsRetriever** class is responsible for retrieving articles from NewsAPI.

### **Functionality**
- Sends requests to the **NewsAPI Everything endpoint** ``
- Filters results by:
  - Topic keyword
  - Language
  - Relevance
- Extracts useful information from each article

### **Returned Information**

Each article includes:
- Title
- Content or description
- Source URL

These articles are later converted into **LangChain Documents** for processing.

### **Error Handling**
If the API request fails, the system displays an error message and safely returns an empty list.

In [ ]:
import requests #HTTP -> NewsAPI
import os
import json

class NewsRetriever:
    def __init__(self):
        # FIX: Access the environment variable name, not the key value itself
        self.api_key = os.environ.get("NEWS_API_KEY")
        self.base_url = "https://newsapi.org/v2/everything"

    def get_articles(self, topic, page_size=5):
        params = {
            "q": topic,
            "language": "en",
            "sortBy": "relevancy",
            "pageSize": page_size,
            "apiKey": self.api_key
        }
        response = requests.get(self.base_url, params=params)
        data = response.json()

        articles = []

        # Error handling for API response
        if response.status_code != 200:
            print(f"Error fetching news: {data.get('message', 'Unknown error')}")
            return articles

        for article in data.get("articles", []):
            content = article.get("content") or article.get("description")
            if content:
                articles.append({
                    "title": article["title"],
                    "content": content,
                    "url": article["url"]
                })
        return articles

## **Embedding and Vector Search Engine**

To enable **semantic search**, articles are transformed into vector embeddings.

### **Embedding Model**
The system uses:

**sentence-transformers/all-MiniLM-L6-v2**

This lightweight transformer model converts text into high-dimensional vectors that capture semantic meaning.

### **Vector Database**
Embeddings are stored in **Chroma**, a vector database designed for similarity search.

### **Workflow**

1. Convert article text into embeddings
2. Store embeddings in Chroma
3. Perform **similarity search** using the user query

This allows the system to retrieve the **most relevant articles**, even if the wording differs from the query.


###**Technical Advantage**: Semantic vs. Lexical Search
Unlike traditional keyword search which relies on exact lexical matching, this system uses **semantic similarity**. For example, a query for "currency trends" can successfully retrieve articles mentioning "dollar prices" or "exchange rates" even if the specific word "trends" is absent, because they share a similar vector space.

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document


class EmbeddingEngine:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        # Added persist_directory for better reliability in Colab
        self.vectorstore = Chroma(embedding_function=self.embeddings)

    def add_articles(self, articles):
        docs = [
            Document(
                page_content=a["content"],
                metadata={"title": a["title"], "url": a["url"]}
            )

            for a in articles
        ]
        self.vectorstore.add_documents(docs)

    def search(self, query, k=3):
        return self.vectorstore.similarity_search(query, k=k)



## **News Summarization Engine**

The **NewsSummarizer** class generates summaries using a large language model.

### **Language Model**
The system uses:

**Kimi K2 Instruct model via Groq API**

This model processes the retrieved documents and produces natural language summaries.

### **Two Summarization Styles**

To satisfy the assignment requirements, the system supports two types of summaries:

| Style | Description |
|------|------|
| Brief | 1–2 sentence summary |
| Detailed | Full paragraph summary |

### **LangChain Summarization Chain**

We utilize LangChain's **"stuff"** chain for its directness and speed. It "stuffs" all retrieved context into a single prompt for the LLM.

This is ideal for our application because our  `pageSize =5` constraint ensures the total token count remains well within the Kimi K2 model's 16,384-token context window, avoiding the computational overhead and cost of fragmented chains like map_reduce.

### **Prompt Engineering**
We implemented custom PromptTemplates. This enforces strict constraint behavior, ensuring the model adheres to the 1-2 sentence **"Brief"** limit or the paragraph **"Detailed"** requirement regardless of the input text length.

In [ ]:
from langchain_groq import ChatGroq #LangChain wrapper
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_core.prompts import PromptTemplate #customization of LLM prompts
class NewsSummarizer:
    def __init__(self):
        self.llm = ChatGroq(
            model_name="moonshotai/kimi-k2-instruct-0905",
            temperature=0.3
        )

    def generate_summary(self, docs, style="brief"):
        # Personalized Prompting
        if style == "brief":
            template = "Write a concise 1-2 sentence summary of these specific news articles:\n\n{text}"
        else:
            template = (
                "Write a detailed, comprehensive paragraph summarizing the key points "
                "of these news articles. Ensure the response is a full, informative paragraph:\n\n{text}"
            )

        prompt = PromptTemplate.from_template(template)
        chain = load_summarize_chain(self.llm, chain_type="stuff", prompt=prompt)

        summary = chain.invoke(docs)["output_text"]

        sources = list(set([doc.metadata.get("url") for doc in docs if doc.metadata.get("url")]))

        return summary, sources



## **User Preference Tracking System**

The **UserManager**class handles persistent data through JSON Serialization. This allows the system to maintain a stateful search history and user preferences without the resource overhead of a relational database.

User data is stored locally in a JSON file called:

`user_data.json`

### **Stored Information**

| Data | Description |
|-----|-----|
| topics | User's saved topics of interest |
| history | Previous search queries |

### **Features**

- Save topics the user wants to follow
- Track search history
- Persist data between application runs

This design provides a **lightweight personalization system** without requiring a database.

In [ ]:
class UserManager:
    def __init__(self, file_path="user_data.json"):
        self.file_path = file_path
        if not os.path.exists(self.file_path):
            with open(self.file_path, "w") as f:
                json.dump({"topics": [], "history": []}, f)

    def load(self):
        with open(self.file_path) as f:
            return json.load(f)

    def save_topic(self, topic):
        data = self.load()
        if topic not in data["topics"]:
            data["topics"].append(topic)
            self._write(data)

    def add_history(self, topic):
        data = self.load()
        data["history"].append(topic)
        self._write(data)

    def get_topics(self):
        return self.load()["topics"]

    def get_history(self):
        return self.load()["history"]

    def _write(self, data):
        with open(self.file_path, "w") as f:
            json.dump(data, f, indent=4)



## **Command Line Interface**

The application provides a **menu-based command line interface** allowing users to interact with the system.

### **Available Options**

1. **Search News**
   - Retrieves articles
   - Generates embeddings
   - Produces a summary

2. **View Saved Topics**
   - Displays the user's saved interests

3. **View History**
   - Shows previous search queries

4. **Exit**
   - Terminates the application

This interface demonstrates how the different components of the system interact within a complete application workflow.

In [ ]:
# --- Main Application Loop ---
retriever = NewsRetriever()
embedder = EmbeddingEngine()
summarizer = NewsSummarizer()
user = UserManager()

while True:
    print("\n=== News App (Powered by Kimi K2) ===")
    print("1. Search News")
    print("2. View Saved Topics")
    print("3. View History")
    print("4. Exit")

    choice = input("Choose option: ")

    if choice == "1":
        topic = input("Enter topic: ")
        articles = retriever.get_articles(topic)

        if not articles:
            print(f"\nNo articles found for '{topic}'.")
            continue

        # Show the user what was fetched (Requirement: Retrieves news articles)
        print(f"\n✅ Fetched {len(articles)} articles. Titles:")
        for i, a in enumerate(articles, 1):
            print(f"   {i}. {a['title']}")

        # Vectorize and Search (Requirement: Generates vector embeddings)
        embedder.add_articles(articles)
        docs = embedder.search(topic)

        # Personalized Summaries (Requirement: User preferences)
        summary_type = input("\nChoose summary style (brief/detailed): ").lower()

        print("\n--- Generating Summary... ---")
        summary = summarizer.generate_summary(docs, style=summary_type)

        print("\n=== SUMMARY ===\n")
        print(summary)

        # Maintenance (Requirement: Maintains user topic preferences)
        user.add_history(topic)
        if input("\nSave this topic to your preferences? (y/n): ").lower() == "y":
            user.save_topic(topic)

    elif choice == "2":
        print("\nSaved Topics:", user.get_topics())

    elif choice == "3":
        print("\nSearch History:", user.get_history())

    elif choice == "4":
        print("Goodbye!")
        break

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=== News App (Powered by Kimi K2) ===
1. Search News
2. View Saved Topics
3. View History
4. Exit

✅ Fetched 5 articles. Titles:
   1. US and Israel Launch Strikes Against Iran
   2. Here’s Why Trump Posted About Iran ‘Stealing’ the 2020 Election Hours After the US Attacked
   3. A new video from the White House mixes Call of Duty footage with actual video of Iran strikes
   4. What Happens if Iran Shuts Down the Strait of Hormuz?
   5. Here’s Every Country Directly Impacted by the War on Iran

--- Generating Summary... ---

=== SUMMARY ===

('U.S. and Israeli forces launched coordinated strikes on Iran, with President Trump declaring it a major combat operation against the Iranian regime, triggering regional turmoil.', ['https://www.wired.com/story/every-country-directly-impacted-by-the-war-on-iran/', 'https://www.wired.com/story/us-iran-strike-donald-trump/', 'https://www.wired.com/story/heres-why-trump-posted-about-iran-stealing-the-2020-election-hours-after-the-us-attacked/'])

==